# Phase 6.9 - controlled market-feature expansion

This student-run notebook evaluates two predeclared groups on the exact Phase 5 lineage. It creates a new dataset version and never overwrites the Phase 5 dataset, the completed continuation reports, or the friend's recovery evidence. Read `docs/PHASE6.md` before running it.

## 1. Prepare the repository and restore the frozen Phase 5 input

In Colab this clones the selected branch and downloads the frozen, Git-ignored Phase 5 dataset without mounting Google Drive. Locally it uses the current repository.

In [ ]:
from pathlib import Path
import subprocess
import sys

BRANCH = 'feature/model-evaluation'
COLAB_REPOSITORY = Path('/content/HalalifyAI')
if 'google.colab' in sys.modules:
    if not (COLAB_REPOSITORY / '.git').is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(COLAB_REPOSITORY)], check=True)
    root = COLAB_REPOSITORY.resolve()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt'), '-r', str(root / 'requirements-cleaning.txt'), 'gdown>=5.2,<6'], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'src' / 'model_training').is_dir():
        root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.notebook_environment import ensure_phase5_dataset, prepare_repository
root = prepare_repository(branch=BRANCH, colab_repository=COLAB_REPOSITORY)
dataset = ensure_phase5_dataset(root)
%cd {root}
print('Project folder:', root)
print('Frozen Phase 5 dataset:', dataset)

## 2. Run the focused safety tests

These tests check formulas, availability, future invariance, missingness, separate versioning, and the precommitment. They do not fit models.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_phase6_expansion_features.py',
    'tests/test_phase6_expanded_dataset.py',
    'tests/test_phase6_market_expansion_contract.py',
], check=True)

## 3. Build and validate the new dataset and protocol

This creates `phase6_expanded_features_v1` and a new protocol while preserving all original rows, labels, splits, eligibility flags, and parent files. It performs no model fitting.

In [ ]:
from src.model_training.feature_ablation.phase6_market_expansion import (
    DATASET, OUTPUT_ROOT, experiment_status, prepare_expansion_inputs,
)
validation = prepare_expansion_inputs()
print('Validation status:', validation['status'])
print('Model fitting performed:', validation['model_fitting_performed'])
print('New dataset:', DATASET)
print('New report root:', OUTPUT_ROOT)
experiment_status()

## 4. Compare cross-sectional relative strength

For return and risk separately, this compares the original 14 features with those same features plus the three peer-rank features on identical rows across all four chronological folds. It refuses to overwrite prior output.

In [ ]:
from src.model_training.feature_ablation.phase6_market_expansion import write_group_comparison
cross_sectional = write_group_comparison('cross_sectional_relative_strength')
for task, summary in cross_sectional['task_summaries'].items():
    print(task, 'decision:', summary['decision'])
    print('  mean difference:', summary['mean_difference'])
    print('  positive folds:', summary['positive_fold_count'], '/ 4')
    if task == 'risk':
        print('  risk guardrails:', summary['risk_guardrails'])

## 5. Compare the SPY market regime group

This repeats the same matched-row, four-fold comparison for the three market-regime features.

In [ ]:
market_regime = write_group_comparison('market_regime')
for task, summary in market_regime['task_summaries'].items():
    print(task, 'decision:', summary['decision'])
    print('  mean difference:', summary['mean_difference'])
    print('  positive folds:', summary['positive_fold_count'], '/ 4')
    if task == 'risk':
        print('  risk guardrails:', summary['risk_guardrails'])

## 6. Confirm completion

Both group flags must be `True`. Do not start Phase 7 yet. First preserve the generated report folder and review both summaries; the final Phase 6 freeze must include only groups that passed the predeclared rule.

In [ ]:
status = experiment_status()
print(status)
assert status['cross_sectional_complete'] and status['market_regime_complete']
print('Phase 6 expansion execution is complete. Preserve and review:', OUTPUT_ROOT)